# LeetCode 414 — Third Maximum Number bằng Python

**Link đề:** https://leetcode.com/problems/third-maximum-number/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting

```python
class Solution:
    def thirdMax(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **giá trị lớn nhất phân biệt** (distinct maximum): các số trùng nhau chỉ tính một lần.
2. Dùng `set` để khử trùng, `sorted(..., reverse=True)`, `max`, `float('-inf')`, `None`.
3. Học kỹ thuật **duy trì top-3 trong một lần duyệt** (giống "cập nhật kỷ lục" ở cuộc thi chạy): đẩy kỷ lục cũ xuống hạng dưới.
4. Viết 4 lời giải: set + sort → xóa max hai lần → ba biến một lần duyệt (Follow up `O(n)`) → heap kích thước 3.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (k-th distinct max).

## 1. Đề bài

Cho mảng số nguyên `nums`. Trả về **giá trị lớn thứ ba phân biệt** trong mảng. Nếu không tồn tại (mảng có ít hơn 3 giá trị khác nhau) thì trả về **giá trị lớn nhất**.

| `nums` | Output | Giải thích |
|---|:---:|---|
| `[3,2,1]` | `1` | phân biệt: 3 > 2 > 1 → thứ ba là 1 |
| `[1,2]` | `2` | chỉ có 2 giá trị khác nhau → trả max = 2 |
| `[2,2,3,1]` | `1` | hai số 2 tính chung một lần: 3 > 2 > 1 |

**Constraints:**
- `1 <= nums.length <= 10^4`
- `-2^31 <= nums[i] <= 2^31 - 1`

**Follow up:** Tìm lời giải `O(n)`?

Câu hỏi đặt ra:
- "Phân biệt" khác "theo vị trí" thế nào? `[2,2,3,1]`: sắp giảm là `[3,2,2,1]`, phần tử thứ ba là `2` — nhưng đáp án là `1`!
- Giá trị có thể là `-2^31` — chọn "giá trị khởi tạo" cho biến max thế nào cho an toàn?

## 2. Nền tảng Python

### 2.1. `set` — tập hợp khử trùng

`set(nums)` giữ mỗi giá trị **một lần**. Thứ tự trong set **không** được đảm bảo, nên muốn thứ tự phải sắp xếp.

In [1]:
nums = [2, 2, 3, 1]
s = set(nums)
print(s, "| số giá trị khác nhau:", len(s))
print(3 in s, 5 in s)

{1, 2, 3} | số giá trị khác nhau: 3
True False


### 2.2. `sorted(..., reverse=True)` và chỉ số

`sorted(x)` trả **list mới** đã sắp tăng dần; `reverse=True` để giảm dần. Phần tử thứ ba là chỉ số `[2]` (vì đếm từ 0).

In [2]:
print(sorted([2, 2, 3, 1], reverse=True))          # theo vị trí: [3, 2, 2, 1]
print(sorted(set([2, 2, 3, 1]), reverse=True))     # phân biệt:   [3, 2, 1]
desc = sorted(set([2, 2, 3, 1]), reverse=True)
print("thứ nhất:", desc[0], "| thứ ba:", desc[2])

[3, 2, 2, 1]
[3, 2, 1]
thứ nhất: 3 | thứ ba: 1


### 2.3. Giá trị "âm vô cùng" và `None`

Khi đi tìm max, ta cần giá trị khởi tạo **nhỏ hơn mọi phần tử**. Có hai lựa chọn an toàn:
- `float('-inf')`: số thực âm vô cùng; mọi số nguyên đều lớn hơn nó.
- `None`: "chưa có giá trị" — phải kiểm tra `x is None` trước khi so sánh.

**Không** dùng `0` hay `-1` hay `-2**31` vì phần tử có thể đúng bằng giá trị đó.

In [3]:
neg_inf = float("-inf")
print(-2**31 > neg_inf, neg_inf < -10**100)
first = None
print("first is None ->", first is None)
try:
    print(None < 5)
except TypeError as e:
    print("So sánh None với số -> TypeError:", e)

True True
first is None -> True
So sánh None với số -> TypeError: '<' not supported between instances of 'NoneType' and 'int'


## 3. Cách trực giác (baseline) — khử trùng rồi sắp xếp

1. `distinct = set(nums)`.
2. Nếu `len(distinct) < 3` → trả `max(distinct)`.
3. Ngược lại sắp giảm dần, trả phần tử chỉ số 2.

In [4]:
def third_max_sort(nums: list) -> int:
    distinct = sorted(set(nums), reverse=True)
    if len(distinct) < 3:
        return distinct[0]
    return distinct[2]

print(third_max_sort([3, 2, 1]))      # 1
print(third_max_sort([1, 2]))         # 2
print(third_max_sort([2, 2, 3, 1]))   # 1

1
2
1


### 3.1. Cách 2 — xóa max hai lần

Không cần sắp toàn bộ: lấy `set`, nếu đủ 3 phần tử thì **xóa max** hai lần, max còn lại chính là thứ ba. Mỗi `max` tốn `O(k)` với `k` số giá trị khác nhau → tổng `O(n)`. Nhưng vẫn tốn `O(n)` bộ nhớ cho set.

In [5]:
def third_max_remove(nums: list) -> int:
    distinct = set(nums)
    if len(distinct) < 3:
        return max(distinct)
    distinct.remove(max(distinct))
    distinct.remove(max(distinct))
    return max(distinct)

print(third_max_remove([3, 2, 1]), third_max_remove([1, 2]), third_max_remove([2, 2, 3, 1]))

1 2 1


## 4. Nút thắt và insight

**Nút thắt:** sắp xếp tốn `O(n log n)`; set tốn `O(n)` bộ nhớ. Follow up đòi `O(n)` — ta muốn **một lần duyệt** và chỉ vài biến.

**Insight — bảng xếp hạng 3 chỗ:** giữ `first > second > third` là **ba giá trị phân biệt lớn nhất đã thấy**. Với mỗi số `x`:
1. Nếu `x` **bằng** một trong ba → bỏ qua (đã có, không được tính lại).
2. Nếu `x > first` → `x` lên hạng 1; hạng 1 cũ xuống hạng 2; hạng 2 cũ xuống hạng 3.
3. Nếu `second < x < first` → `x` lên hạng 2; hạng 2 cũ xuống hạng 3.
4. Nếu `third < x < second` → `x` thay hạng 3.

Giống bảng xếp hạng kỷ lục: ai phá kỷ lục thì những người dưới **dồn xuống một bậc**.

In [6]:
# Minh họa "dồn xuống" bằng gán đồng thời
first, second, third = 5, 3, 1
x = 7
first, second, third = x, first, second     # vế phải được tính TRƯỚC rồi mới gán
print(first, second, third)                 # 7 5 3

7 5 3


Gán đồng thời `a, b, c = x, a, b` rất tiện: Python tính **toàn bộ vế phải trước** (dùng giá trị cũ của `a`, `b`) rồi mới gán. Không cần biến tạm.

## 5. Lời giải chính — ba biến, một lần duyệt

**Đoạn 1 — khởi tạo:** `first = second = third = None` (chưa có ai trên bảng).

**Đoạn 2 — bỏ qua trùng:** `if x == first or x == second or x == third: continue`.

**Đoạn 3 — xếp hạng** (chú ý kiểm tra `None` trước khi so sánh):
```python
if first is None or x > first:
    first, second, third = x, first, second
elif second is None or x > second:
    second, third = x, second
elif third is None or x > third:
    third = x
```
**Đoạn 4 — kết luận:** `return first if third is None else third`.

In [7]:
def third_max_three_vars(nums: list) -> int:
    first = second = third = None
    for x in nums:
        if x == first or x == second or x == third:
            continue
        if first is None or x > first:
            first, second, third = x, first, second
        elif second is None or x > second:
            second, third = x, second
        elif third is None or x > third:
            third = x
    return first if third is None else third

print(third_max_three_vars([3, 2, 1]), third_max_three_vars([1, 2]), third_max_three_vars([2, 2, 3, 1]))
print(third_max_three_vars([1, 2, -2**31]))     # -2147483648 là thứ ba hợp lệ

1 2 1
-2147483648


Vì sao `x == first` an toàn khi `first is None`? Vì `5 == None` chỉ đơn giản trả `False` (khác với `<`, `>` gây lỗi). Do đó đoạn 2 đúng cả khi bảng chưa đầy.

## 6. TRACE từng bước

Ví dụ `nums = [2, 2, 3, 1, 5, 3]`:

| x | trùng? | Nhánh | first | second | third |
|:---:|:---:|---|:---:|:---:|:---:|
| 2 | không | first None | 2 | None | None |
| 2 | có | bỏ qua | 2 | None | None |
| 3 | không | x > first | 3 | 2 | None |
| 1 | không | third None | 3 | 2 | 1 |
| 5 | không | x > first | 5 | 3 | 2 |
| 3 | có | bỏ qua | 5 | 3 | 2 |

Kết quả `third = 2`.

In [8]:
def trace_third_max(nums: list) -> int:
    first = second = third = None
    print(f"{'x':>4} | {'hành động':<22} | first, second, third")
    for x in nums:
        if x == first or x == second or x == third:
            action = "trùng -> bỏ qua"
        elif first is None or x > first:
            first, second, third = x, first, second
            action = "lên hạng 1, dồn xuống"
        elif second is None or x > second:
            second, third = x, second
            action = "lên hạng 2, dồn xuống"
        elif third is None or x > third:
            third = x
            action = "thay hạng 3"
        else:
            action = "nhỏ hơn hạng 3 -> bỏ"
        print(f"{x:>4} | {action:<22} | {first}, {second}, {third}")
    res = first if third is None else third
    print("Kết quả:", res)
    return res

trace_third_max([2, 2, 3, 1, 5, 3])

   x | hành động              | first, second, third
   2 | lên hạng 1, dồn xuống  | 2, None, None
   2 | trùng -> bỏ qua        | 2, None, None
   3 | lên hạng 1, dồn xuống  | 3, 2, None
   1 | thay hạng 3            | 3, 2, 1
   5 | lên hạng 1, dồn xuống  | 5, 3, 2
   3 | trùng -> bỏ qua        | 5, 3, 2
Kết quả: 2


2

### 6.1. Trace nhánh "không đủ 3" và số cực tiểu

`[1, 2]` → `third` vẫn `None` → trả `first = 2`. `[1, 1, -2**31]`: chỉ có 2 giá trị phân biệt → trả `1`.

In [9]:
trace_third_max([1, 2])
print()
trace_third_max([1, 1, -2**31])

   x | hành động              | first, second, third
   1 | lên hạng 1, dồn xuống  | 1, None, None
   2 | lên hạng 1, dồn xuống  | 2, 1, None
Kết quả: 2

   x | hành động              | first, second, third
   1 | lên hạng 1, dồn xuống  | 1, None, None
   1 | trùng -> bỏ qua        | 1, None, None
-2147483648 | lên hạng 2, dồn xuống  | 1, -2147483648, None
Kết quả: 1


1

## 7. Cách 5 — heap (đống) kích thước 3

`heapq` là module chuẩn cài **min-heap**: phần tử nhỏ nhất luôn ở `heap[0]`; `heappush` / `heappop` tốn `O(log k)`.

Ý tưởng: giữ heap chứa **tối đa 3 giá trị phân biệt lớn nhất**. Thêm `x` (nếu chưa có); khi heap > 3 phần tử → `heappop` bỏ phần tử nhỏ nhất. Cuối cùng nếu heap đủ 3 thì `heap[0]` là thứ ba, ngược lại trả `max(heap)`.

Kỹ thuật này tổng quát lên **k** bất kỳ: `O(n log k)`.

In [10]:
import heapq

h = []
for v in [5, 1, 8, 3]:
    heapq.heappush(h, v)
    print("push", v, "-> heap[0] (nhỏ nhất) =", h[0])
print("pop ->", heapq.heappop(h), "| còn", sorted(h))

push 5 -> heap[0] (nhỏ nhất) = 5
push 1 -> heap[0] (nhỏ nhất) = 1
push 8 -> heap[0] (nhỏ nhất) = 1
push 3 -> heap[0] (nhỏ nhất) = 1
pop -> 1 | còn [3, 5, 8]


In [11]:
def third_max_heap(nums: list, k: int = 3) -> int:
    heap = []
    in_heap = set()
    for x in nums:
        if x in in_heap:
            continue
        heapq.heappush(heap, x)
        in_heap.add(x)
        if len(heap) > k:
            in_heap.remove(heapq.heappop(heap))
    return heap[0] if len(heap) == k else max(heap)

print(third_max_heap([3, 2, 1]), third_max_heap([1, 2]), third_max_heap([2, 2, 3, 1]))

1 2 1


## 8. Code nộp LeetCode

Bản chính: ba biến (đúng Follow up `O(n)` thời gian, `O(1)` bộ nhớ). Bản phụ: set + sort ngắn gọn.

In [12]:
from typing import List

class Solution:
    def thirdMax(self, nums: List[int]) -> int:
        first = second = third = None
        for x in nums:
            if x == first or x == second or x == third:
                continue
            if first is None or x > first:
                first, second, third = x, first, second
            elif second is None or x > second:
                second, third = x, second
            elif third is None or x > third:
                third = x
        return first if third is None else third


sol = Solution()
print(sol.thirdMax([3, 2, 1]), sol.thirdMax([1, 2]), sol.thirdMax([2, 2, 3, 1]))

1 2 1


In [13]:
class SolutionSort:
    def thirdMax(self, nums: List[int]) -> int:
        d = sorted(set(nums), reverse=True)
        return d[2] if len(d) >= 3 else d[0]

print(SolutionSort().thirdMax([2, 2, 3, 1]))

1


## 9. Vì sao đúng? (invariant)

**Bất biến:** sau khi xử lý tiền tố `nums[:i]`, các biến không `None` trong `(first, second, third)` chính là **các giá trị phân biệt lớn nhất** của tiền tố, xếp **giảm ngặt**: `first > second > third`.

- Ban đầu tiền tố rỗng, cả ba `None` → đúng.
- Xét `x`:
  - Nếu `x` đã nằm trong bảng → tập giá trị phân biệt top-3 không đổi.
  - Nếu `x` không nằm trong bảng, nó lớn hơn hạng nào đó (hoặc có chỗ trống) → chèn vào đúng vị trí, các hạng dưới dồn xuống, phần tử thứ tư (nếu có) bị đẩy ra — chính là phần tử không còn thuộc top-3.
  - Nếu `x` nhỏ hơn cả ba hạng đã đầy → không vào top-3 → bỏ qua đúng.
- Cuối cùng: nếu `third` không `None` thì có ít nhất 3 giá trị phân biệt và `third` là lớn thứ ba; ngược lại trả `first` = max. ∎

Thứ tự kiểm tra "trùng" **trước** "xếp hạng" là bắt buộc: nếu không, `x == first` sẽ rơi vào nhánh `x > second` và tạo bản sao.

## 10. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|:---:|:---:|
| set + sort | `O(n log n)` | `O(n)` |
| set + xóa max hai lần | `O(n)` | `O(n)` |
| **Ba biến** | `O(n)` | `O(1)` |
| Heap kích thước k (+ set nhỏ) | `O(n log k)` = `O(n)` với k=3 | `O(k)` |

Ba biến là lời giải tối ưu cho Follow up. Heap tổng quát hơn khi `k` thay đổi.

## 11. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<36} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", [3, 2, 1], 1),
    ("đề 2: thiếu -> max", [1, 2], 2),
    ("đề 3: trùng tính một lần", [2, 2, 3, 1], 1),
    ("một phần tử", [7], 7),
    ("toàn trùng", [5, 5, 5, 5], 5),
    ("chứa -2^31 là đáp án", [1, 2, -2**31], -2**31),
    ("chứa -2^31 nhưng thiếu", [1, 1, -2**31], 1),
    ("tăng dần dài", list(range(10)), 7),
    ("giảm dần có trùng", [5, 5, 4, 4, 3, 3, 2], 3),
    ("số âm", [-1, -2, -3, -4], -3),
    ("max ở cuối", [1, 2, 2, 5, 3, 5], 2),
]
for name, nums, exp in cases:
    check(name, sol.thirdMax(nums), exp)

PASS  đề 1                                 -> 1
PASS  đề 2: thiếu -> max                   -> 2
PASS  đề 3: trùng tính một lần             -> 1
PASS  một phần tử                          -> 7
PASS  toàn trùng                           -> 5
PASS  chứa -2^31 là đáp án                 -> -2147483648
PASS  chứa -2^31 nhưng thiếu               -> 1
PASS  tăng dần dài                         -> 7
PASS  giảm dần có trùng                    -> 3
PASS  số âm                                -> -3
PASS  max ở cuối                           -> 2


In [15]:
import random
rng = random.Random(414)
funcs = [third_max_sort, third_max_remove, third_max_three_vars, third_max_heap,
         Solution().thirdMax, SolutionSort().thirdMax]
for _ in range(2000):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 12))]
    exp = third_max_sort(nums)
    for f in funcs:
        assert f(nums) == exp, (f.__name__, nums)
big = [rng.randint(-2**31, 2**31 - 1) for _ in range(10 ** 4)]
assert len({f(big) for f in funcs}) == 1
print("2000 mảng nhỏ + 1 mảng 10^4 phần tử: 6 cách đồng thuận -> PASS")

2000 mảng nhỏ + 1 mảng 10^4 phần tử: 6 cách đồng thuận -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Không khử trùng
```python
return sorted(nums, reverse=True)[2]
```
`[2,2,3,1]` → `2` (sai, đáp án 1). Cũng lỗi `IndexError` khi mảng có < 3 phần tử.

### Lỗi 2: Khởi tạo bằng số "đủ nhỏ" như `-2**31`
Nếu `third = -2**31` ban đầu, với `[1, 2, -2**31]` ta không phân biệt được "chưa có hạng 3" và "hạng 3 là -2^31" → trả `2` sai (đáp án `-2147483648`).

In [16]:
def wrong_no_dedupe(nums):
    d = sorted(nums, reverse=True)
    return d[2] if len(d) >= 3 else d[0]

def wrong_sentinel(nums):
    MIN = -2 ** 31
    first = second = third = MIN
    for x in nums:
        if x in (first, second, third):
            continue
        if x > first:
            first, second, third = x, first, second
        elif x > second:
            second, third = x, second
        elif x > third:
            third = x
    return first if third == MIN else third

print("lỗi 1 [2,2,3,1]:", wrong_no_dedupe([2, 2, 3, 1]), "(đúng 1)")
print("lỗi 2 [1,2,-2^31]:", wrong_sentinel([1, 2, -2**31]), "(đúng", -2**31, ")")

lỗi 1 [2,2,3,1]: 2 (đúng 1)
lỗi 2 [1,2,-2^31]: 2 (đúng -2147483648 )


### Lỗi 3: Quên bước "bỏ qua trùng"
Với `[2, 2, 1]`: số `2` thứ hai lọt vào nhánh `x > second` (second đang `None`) → bảng thành `2, 2, ...` → trả sai.

### Lỗi 4: Dồn xuống sai thứ tự (gán tuần tự)
```python
first = x
second = first      # lúc này first đã là x!
third = second
```
Phải gán **đồng thời** hoặc gán từ dưới lên: `third = second; second = first; first = x`.

In [17]:
def wrong_no_skip(nums):
    first = second = third = None
    for x in nums:
        if first is None or x > first:
            first, second, third = x, first, second
        elif second is None or x > second:
            second, third = x, second
        elif third is None or x > third:
            third = x
    return first if third is None else third

def wrong_sequential(nums):
    first = second = third = None
    for x in nums:
        if x in (first, second, third):
            continue
        if first is None or x > first:
            first = x; second = first; third = second      # sai thứ tự
        elif second is None or x > second:
            second = x; third = second
        elif third is None or x > third:
            third = x
    return first if third is None else third

print("lỗi 3 [2,2,1]:", wrong_no_skip([2, 2, 1]), "(đúng 2)")
print("lỗi 4 [1,2,3]:", wrong_sequential([1, 2, 3]), "(đúng 1)")

lỗi 3 [2,2,1]: 1 (đúng 2)
lỗi 4 [1,2,3]: 3 (đúng 1)


### Lỗi 5: Dùng `if` thay cho `elif` ở các nhánh xếp hạng
Một số `x` có thể bị xếp vào **nhiều** hạng cùng lúc. Các nhánh phải loại trừ nhau.

## 13. Mẹo nhận dạng pattern

- "Lớn thứ k / nhỏ thứ k" với `k` **nhỏ, cố định** → giữ `k` biến và "dồn xuống".
- `k` thay đổi / lớn → **min-heap kích thước k** (`O(n log k)`), hoặc Quickselect (`O(n)` trung bình).
- Có chữ "**distinct**" → khử trùng (set) hoặc kiểm tra bằng nhau trước khi chèn.
- Phần tử có thể chạm biên kiểu số → khởi tạo bằng `None` hoặc `float('-inf')`, đừng dùng "số đủ nhỏ".

## 14. Biến thể

1. **k-th distinct maximum** — `third_max_heap(nums, k)`.
2. **Second largest** — chỉ cần 2 biến.
3. **LeetCode 215 — Kth Largest Element in an Array** (không khử trùng!) → heap không cần set.
4. **Ba số lớn nhất kèm số lần xuất hiện.**

In [18]:
def kth_distinct_max(nums, k):
    d = sorted(set(nums), reverse=True)
    return d[k - 1] if len(d) >= k else d[0]

def kth_largest_with_dups(nums, k):
    return heapq.nlargest(k, nums)[-1]

nums = [3, 2, 3, 1, 2, 4, 5, 5, 6]
print("2nd distinct:", kth_distinct_max(nums, 2), "| 4th distinct:", kth_distinct_max(nums, 4))
print("4th largest (có trùng, LC 215):", kth_largest_with_dups(nums, 4))
print("so với heap của mình k=4:", third_max_heap(nums, 4))

2nd distinct: 5 | 4th distinct: 3
4th largest (có trùng, LC 215): 4
so với heap của mình k=4: 3


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** Trace bảng `first, second, third` cho `nums = [5, 1, 5, 2, 4, 3]`.

**Bài 2 (đoán output):** `Solution().thirdMax([1, 2, 2, 2])`? Và `[-1, -1, -1, 0, 1]`?

**Bài 3 (điền chỗ trống):** hoàn thiện ba biến.

In [19]:
def exercise_third_max(nums: list) -> int:
    first = second = third = None
    for x in nums:
        # TODO 1: bỏ qua nếu trùng
        # TODO 2: ba nhánh xếp hạng với dồn xuống
        pass
    # TODO 3: trả third hoặc first
    pass

# assert exercise_third_max([2, 2, 3, 1]) == 1
# assert exercise_third_max([1, 2]) == 2

**Bài 4 (tự viết):** `exercise_second_max(nums)` trả giá trị lớn thứ hai phân biệt, không có thì trả `None`. Một lần duyệt, `O(1)` bộ nhớ.

**Bài 5 (biến thể):** `exercise_third_min(nums)` trả giá trị **nhỏ** thứ ba phân biệt (không có thì trả min).

**Bài 6 (mini challenge):** `exercise_top3_distinct(nums)` trả list tối đa 3 giá trị phân biệt lớn nhất theo thứ tự giảm, **không** dùng `sort`/`set`.

In [20]:
def exercise_second_max(nums: list):
    # TODO
    pass

def exercise_third_min(nums: list) -> int:
    # TODO
    pass

def exercise_top3_distinct(nums: list) -> list:
    # TODO
    pass

# assert exercise_second_max([3, 3, 1]) == 1 and exercise_second_max([4, 4]) is None
# assert exercise_third_min([2, 2, 3, 1]) == 3
# assert exercise_top3_distinct([2, 2, 3, 1, 5]) == [5, 3, 2]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Làm từng bước như mục 6 | 5 trùng thì bỏ | Kết quả cuối `5, 4, 3` |
| 2 | Đếm giá trị phân biệt | `{1,2}` chỉ 2 giá trị | `{-1,0,1}` đủ 3 |
| 3 | Theo mục 5 | `continue` khi trùng | `first, second, third = x, first, second` |
| 4 | Như bài chính nhưng 2 biến | `first, second = x, first` | Trả `second` (có thể `None`) |
| 5 | Đảo dấu so sánh | `x < first` ... | Hoặc gọi `thirdMax` trên `[-x for x in nums]` rồi đổi dấu |
| 6 | Dùng lại ba biến | Gom các biến khác `None` | `[v for v in (first, second, third) if v is not None]` |

## 17. Đáp án tham khảo

- **Bài 1:** `5 → (5,·,·)`, `1 → (5,1,·)`, `5` trùng, `2 → (5,2,1)`, `4 → (5,4,2)`, `3 → (5,4,3)` → **3**.
- **Bài 2:** `[1,2,2,2]` chỉ 2 giá trị phân biệt → **2**; `[-1,-1,-1,0,1]` → `1 > 0 > -1` → **-1**.

In [21]:
trace_third_max([5, 1, 5, 2, 4, 3])
print("Bài 2:", Solution().thirdMax([1, 2, 2, 2]), Solution().thirdMax([-1, -1, -1, 0, 1]))

   x | hành động              | first, second, third
   5 | lên hạng 1, dồn xuống  | 5, None, None
   1 | lên hạng 2, dồn xuống  | 5, 1, None
   5 | trùng -> bỏ qua        | 5, 1, None
   2 | lên hạng 2, dồn xuống  | 5, 2, 1
   4 | lên hạng 2, dồn xuống  | 5, 4, 2
   3 | thay hạng 3            | 5, 4, 3
Kết quả: 3
Bài 2: 2 -1


In [22]:
def answer_third_max(nums: list) -> int:
    first = second = third = None
    for x in nums:
        if x == first or x == second or x == third:
            continue
        if first is None or x > first:
            first, second, third = x, first, second
        elif second is None or x > second:
            second, third = x, second
        elif third is None or x > third:
            third = x
    return first if third is None else third

def answer_second_max(nums: list):
    first = second = None
    for x in nums:
        if x == first or x == second:
            continue
        if first is None or x > first:
            first, second = x, first
        elif second is None or x > second:
            second = x
    return second

def answer_third_min(nums: list) -> int:
    return -answer_third_max([-x for x in nums])

def answer_top3_distinct(nums: list) -> list:
    first = second = third = None
    for x in nums:
        if x == first or x == second or x == third:
            continue
        if first is None or x > first:
            first, second, third = x, first, second
        elif second is None or x > second:
            second, third = x, second
        elif third is None or x > third:
            third = x
    return [v for v in (first, second, third) if v is not None]

assert answer_third_max([2, 2, 3, 1]) == 1 and answer_third_max([1, 2]) == 2
assert answer_second_max([3, 3, 1]) == 1 and answer_second_max([4, 4]) is None
assert answer_third_min([2, 2, 3, 1]) == 3 and answer_third_min([5, 7]) == 5
assert answer_top3_distinct([2, 2, 3, 1, 5]) == [5, 3, 2] and answer_top3_distinct([4]) == [4]
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def thirdMax(self, nums: List[int]) -> int:
        first = second = third = None
        for x in nums:
            if x == first or x == second or x == third:   # distinct
                continue
            if first is None or x > first:
                first, second, third = x, first, second      # dồn xuống
            elif second is None or x > second:
                second, third = x, second
            elif third is None or x > third:
                third = x
        return first if third is None else third
```

**Checklist:**
- [ ] Khử trùng ("distinct") — kiểm tra bằng nhau trước khi xếp hạng.
- [ ] Không dùng sentinel `-2**31` / `0`; dùng `None` hoặc `-inf`.
- [ ] Dồn xuống bằng **gán đồng thời**.
- [ ] Các nhánh xếp hạng dùng `elif`.
- [ ] Thiếu hạng 3 → trả **max**, không phải `None`.
- [ ] Time `O(n)`, Space `O(1)`.